# A passive neuron as a low-pass filter

The neuron model is based on the RC circuit in NEURON.

This notebook grades your answers for you, and **each student gets a slightly
different membrane** -- so the cut-off frequency you find below is your own.

## Step 1: Setup

In [ ]:
# Setup inline plotting
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# For Google Colab, this line installs NEURON and quantities
#!pip install neuron quantities

In [ ]:
# We will let this library handle unit conversion for us
import quantities as pq
from quantities import um, nS, mV, cm, ms, nA, S, uF, Hz, degrees, s

In [ ]:
# Import and initialize NEURON
import neuron
from neuron import h
h.load_file("stdrun.hoc")

In [ ]:
# Import other modules we need
import numpy as np

## Step 1b: Load your personal exercise parameters

This notebook will grade your answers for you, and **each student gets a slightly
different membrane**. The cell below fetches your own soma length and leak
conductance; the rest of the notebook builds the circuit from them, so the
cut-off frequency and the answers you submit at the end are specific to the
membrane you simulated.

`grading.load()` reads the launch file the platform writes next to this notebook.
You never handle any tokens or URLs yourself.

In [ ]:
from obi_notebook import grading

assignment = grading.load()

# Your personal parameters for this assignment.
soma_length = assignment.params["soma_length_um"]
g_leak = assignment.params["g_leak_nS"]
f_probe = assignment.params["probe_frequency_hz"]  # used in Question 5

print(f"Your soma length:      {soma_length} um")
print(f"Your leak conductance: {g_leak} nS")
print(f"Your probe frequency:  {f_probe} Hz")
print(f"Exercises to submit:   {assignment.exercise_keys}")

## Step 2: Define the circuit
We will use a single compartment, called a "Section" (more on that in next lectures). <br>
It has a cylindrical geometry with length "L" and a diameter "diam", and a specific capacitance "cm" (capacitance per area) <br>
**Unit conversion is a common source of error, so we will be explicit with our units.** 

In [ ]:
soma = h.Section()

### Query NEURON for the expected units for soma.L & soma.diam

In [ ]:
[h.units(x) for x in ["L", "diam"]]

In [ ]:
# The units are not required, but making them explicit is good practice.
# soma.L is YOUR personal value, loaded in Step 1b above.
soma.L = soma_length * um
soma.diam = 30 * um

In [ ]:
volume = soma(0.5).volume() * um**3

In [ ]:
area = soma(0.5).area() * um**2

### Assign the membrane capacitance "everywhere"

In [ ]:
h.units("cm")  # Query the expected units

In [ ]:
specific_membrane_capacitance = 1 * uF/cm**2

In [ ]:
for sec in soma.wholetree():
    sec.cm = specific_membrane_capacitance #  specific membrane capacitance (micro Farads / cm^2)

### Add (insert) a leak conductance G = 1/R

In [ ]:
soma.insert("pas")

In [ ]:
h.units("g_pas")

In [ ]:
G = g_leak*nS  # R = 1/G in our RC circuit -- YOUR personal value, see Step 1b

In [ ]:
v_rest = -70*mV

In [ ]:
# Assign the leak conductance everywhere
for seg in soma:
    seg.pas.g = (G/area).rescale(S/cm**2)  # Compute specific conductance, and rescale to units of 'S/cm2'
    seg.pas.e = v_rest

### Add a current injection

In [ ]:
stim = h.IClamp(soma(0.5))

In [ ]:
stim.delay = 0 * ms  # Start injecting current at the start of the simulation
stim.dur = 1500 * ms  # Keep injecting for the whole 1500ms
stim.amp = 0.1 * nA  # Overwritten below by the sinusoid, see .play()

### Add a sinusoidal current of frequency $f$

In [ ]:
t_current = np.arange(0,1500,h.dt)*ms

In [ ]:
# Using pq.sin instead of np.sin handles time units for you 
f = 2*Hz
I_amp = 0.1 * nA
I_t = I_amp * pq.sin(2*np.pi*f*t_current)

In [ ]:
plt.plot(t_current, I_t)
plt.xlabel("time (ms)")
plt.ylabel("current (nA)")

In [ ]:
v_I_t = h.Vector(I_t)
v_t_current = h.Vector(t_current)

In [ ]:
# Apply the time-varying current to the IClamp amplitude parameter
v_I_t.play(stim._ref_amp, v_t_current, True)

## Step 3: Run the simulation

### Define recordings of simulation variables

In [ ]:
soma_v = h.Vector().record(soma(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

### Set the initial voltage

In [ ]:
h.finitialize( float(v_rest) )

### Run the simulation for 1500ms

In [ ]:
h.continuerun( float(1500 * ms) )

In [ ]:
# What would be the voltage trace if it relaxed immediately to V_infinity (i.e. tau->0)
# This can be computed by I*R_input + V_rest as discussed in the lecture.
adiabatic_V = (I_t/G).rescale(mV)+v_rest

## Step 4: Plot the results

In [ ]:
plt.plot(t, soma_v, lw=2, label="soma(0.5).v")
plt.plot(t_current, adiabatic_V, 'r--', lw=2, label="adiabatic V")
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("v [mV]", size=16)
plt.xticks(size=12)
plt.yticks(size=12)
plt.title("Current injection at f=%f Hz" % f)

In [ ]:
# A first look at the amplitude of the oscillation (half the peak-to-peak swing).
# Careful: this includes the startup transient from v_rest -- see compute_amp_ratio below.
(np.max(soma_v.to_python()) - np.min(soma_v.to_python()))/2

In [ ]:
# Computing adiabatic amplitude of voltage oscillations from input resistance Rin = 1/G
(I_amp / G).rescale(mV)

In [ ]:
def compute_amp_ratio(soma_v, t, adiabatic_v_amp, t_settle=500*ms):
    """ Amplitude of the settled v_m oscillation, as a fraction of I_amp * R_in """
    # The membrane starts at v_rest, so the first few tau are a transient on top
    # of the oscillation. Discard them, or they inflate the measured amplitude.
    arr_t, arr_v = np.array(t), np.array(soma_v)
    settled = arr_v[arr_t >= float(t_settle)]
    soma_amp = (np.max(settled) - np.min(settled))/2 * mV
    return float(soma_amp / adiabatic_v_amp)

In [ ]:
def compute_phase_lag(soma_v, t, f, t_settle=500*ms):
    """ Return phase lag of v_m behind the injected current, in degrees """
    arr_t, arr_v = np.array(t), np.array(soma_v)
    period = float((1/f).rescale(ms))
    # Find the voltage peak in the first full period after the transient
    start = int(np.searchsorted(arr_t, float(t_settle)))
    stop = int(np.searchsorted(arr_t, float(t_settle) + period))
    t_peak = arr_t[start + int(np.argmax(arr_v[start:stop]))]
    # I(t) = I_amp * sin(2*pi*f*t) peaks a quarter of the way into each cycle, so
    # the lag is however much further into its own cycle the voltage peak sits.
    cycles = t_peak * float(f.rescale(Hz)) / 1000.0
    return ((cycles - 0.25) % 1.0) * 360 * degrees

In [ ]:
compute_amp_ratio(soma_v, t, (I_amp / G).rescale(mV))

There is a nice agreement between the amplitude of membrane voltage fluctuation, and the adiabatic amplitude given by $I(t)*R_{in}$, as can be seen in the plot above!

What about the phase of the oscillations?

In [ ]:
period = (1/f).rescale(ms)

In [ ]:
period

In [ ]:
compute_phase_lag(soma_v, t, f)

The voltage oscillations follow closely the adiabatic oscillation given by $I(t)*R_{in}$, as can be seen in the plot above!

### Now let's increase f. Can the membrane "follow" higher frequencies? 

In [ ]:
# Using pq.sin instead of np.sin handles time units for you 
f = 50*Hz
I_amp = 0.1 * nA
I_t = I_amp * pq.sin(2*np.pi*f*t_current)

In [ ]:
# Preserve v_I_t object in memory, but fill it with new current values
v_I_t.from_python(I_t)

In [ ]:
h.finitialize( float(v_rest) )
h.continuerun( float(1500 * ms) )

In [ ]:
adiabatic_V = (I_t/G).rescale(mV)+v_rest

In [ ]:
plt.plot(t, soma_v, lw=2, label="soma(0.5).v")
plt.plot(t_current, adiabatic_V, 'r--', lw=2, label="adiabatic V")
plt.xlim(0, 200)
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("v [mV]", size=16)
plt.xticks(size=12)
plt.yticks(size=12)
plt.title("Current injection at f=%f Hz" % f)

In [ ]:
compute_amp_ratio(soma_v, t, (I_amp / G).rescale(mV))

In [ ]:
period = (1/f).rescale(ms)

In [ ]:
period

In [ ]:
compute_phase_lag(soma_v, t, f)

Now the voltage oscillations are having trouble keeping up! They trail the adiabatic oscillation given by $I(t)*R_{in}$ by more than an eighth of a period (45 degrees), and their amplitude is down to roughly half of it, as can be seen in the plot above.

### Cut-off frequency of a low-pass filter

The cut-off fequency, $f_c$, is defined as the frequency where the amplitude of the $v_m$ oscillations is 70.7% of the adiabatic voltage $I_{stim}*R_{in}$ and where the phase of the oscillations lags by 1/8th of an oscillation period (45 degrees or $\pi/4$).

From circuit analysis, we expect that $f_c = 1/2\pi \tau_m$

More generally, circuit analysis gives the response at **any** frequency -- the
two curves you have just sampled at 2 Hz and at 50 Hz:

$$\frac{V_m(f)}{I_{amp}R_{in}} = \frac{1}{\sqrt{1 + (f/f_c)^2}} \qquad\qquad \mathrm{phase\ lag}(f) = \arctan\left(\frac{f}{f_c}\right)$$

Putting $f = f_c$ into these two expressions gives $1/\sqrt{2} = 70.7\%$ and
$\arctan(1) = 45$ degrees -- the definition above.

![low-pass filter diagram could not be fetched](https://www.electronics-tutorials.ws/wp-content/uploads/2018/05/filter-fil10.gif)

In [ ]:
# Converting a voltage ratio of 70.7% to dB
20 * np.log10(0.707)

---

## Now it's your turn! Questions with answer feedback

Answer each question for **your** membrane -- the one you just simulated, using the
parameters printed in Step 1b. Each `submit` call grades one answer and leaves a record in StudiUM; you can re-run a cell to resubmit a better answer.

Answers are scored on relative error: within 5% earns full credit, and credit
fades to zero at 20% off. Three significant figures is plenty.

### Question 1 -- The membrane time constant

What is the membrane time constant $\tau_m$ of **your** membrane, in **ms**?

The two cells below get you there. `g_L` is the *specific* leak conductance that
NEURON actually stores, and dividing the specific capacitance by it gives
$\tau_m = c_m / g_L = C_m / G$ -- the area cancels, so you never need it here.

In [ ]:
g_L = (G/area).rescale(S/cm**2)

In [ ]:
g_L

In [ ]:
tau_m = (specific_membrane_capacitance/g_L).rescale(ms)

In [ ]:
# You computed this just above as `tau_m`.
# Test submitting it!
tau_m_answer = float(tau_m.rescale(ms))

print(f"Submitting {tau_m_answer:.4g} ms")
assignment.submit(tau_m_answer, "tau_m")

### Question 2 -- The cut-off frequency

Find the cut-off frequency of **your** membrane using $f_c = 1/2\pi \tau_m$, in **Hz**.

In [ ]:
# You now have tau_m. Watch the units: f_c = 1/(2*pi*tau_m) is in Hz only if
# tau_m is in seconds. With `quantities`, .rescale(Hz) will do it for you.
f_c = None  # <-- replace with your answer, in Hz

assignment.submit(f_c, "f_cutoff")

### Question 3 -- Amplitude of the oscillation at 2 Hz

At 2 Hz you are far **below** $f_c$, and you saw the membrane follow the injected
current almost perfectly. What is the **amplitude** of the $v_m$ oscillation at
2 Hz, in **mV**?

Amplitude means half the peak-to-peak swing -- the quantity `compute_amp_ratio`
divides by. You can read it off your 2 Hz simulation, or predict it: below $f_c$
the membrane is essentially adiabatic, so the amplitude is just $I_{amp}R_{in}$
with $R_{in} = 1/G$.

In [ ]:
v_amp_answer = None  # <-- replace with your answer, in mV

assignment.submit(v_amp_answer, "v_amp_low_f")

### Question 4 -- Attenuation at 50 Hz

At 50 Hz you are **above** $f_c$ and the oscillation is visibly smaller. What is
the **attenuation** $V_m / (I_{amp}R_{in})$ at 50 Hz?

Report the plain ratio -- a number between 0 and 1, not a percentage and not dB.
`compute_amp_ratio` returns exactly this quantity, and the expression above
predicts it.

In [ ]:
attenuation_answer = None  # <-- replace with your answer, a ratio in [0, 1]

assignment.submit(attenuation_answer, "attenuation_high_f")

### Question 5 -- The whole frequency response

Plot the attenuation $V_m / (I_{amp}R_{in})$ and the phase lag for a range of
frequencies around $f_c$. Does an attenuation of $\approx 70.7\%$ and a phase lag
of 1/8th of a period occur at $f_c$? Visualize the voltage traces for $f = f_c$,
$f = 1/10 * f_c$ (low frequency) and $f = 10 * f_c$ (high frequency).<br>
<br>
**Hint**: You should use a *for* loop, and the **compute_amp_ratio** and
**compute_phase_lag** functions above. For re-running the simulation, follow the
example "Now let's increase f."

**Note**: each frequency costs one simulation, so keep the list short -- 15 to 30
frequencies spread between $f_c/10$ and $10f_c$ is plenty. Start above 0 Hz: a
phase lag is not defined at $f = 0$.

Then read the **phase lag at your own `f_probe`** (printed in Step 1b) off your
curve -- or predict it with the $\arctan$ expression above -- and submit it in
**degrees**.

**Hint**: Below is an example of a for loop collecting values in a list, and plotting them

In [ ]:
# A function to square a number
def compute_square(x):
    return x*x

In [ ]:
# Make an empty list to store values
result_list = []

In [ ]:
# Frequency range
freqs = np.arange(0,100,0.1)

In [ ]:
for x in freqs:
    result = compute_square(x)
    # Append the result to the result_list
    result_list.append(result) 

In [ ]:
# plot(x, y)
plt.plot(freqs, result_list)

In [ ]:
# The phase lag at YOUR probe frequency f_probe, printed in Step 1b.
phase_lag_answer = None  # <-- replace with your answer, in degrees

assignment.submit(phase_lag_answer, "phase_lag_probe")

### Question 6 -- What does the leak conductance control?

Go back to Step 2 and set the leak conductance to **twice** your value:

```python
G = 2 * g_leak * nS
```

Re-run the simulation at 2 Hz and at 50 Hz, then compare against the traces you
had before and convince yourself of two things:

- the low-frequency amplitude $I_{amp}R_{in}$ is **halved** -- the membrane is
  less sensitive to the injected current
- but $\tau_m = C_m/G$ is halved too, so the cut-off frequency $f_c = 1/2\pi\tau_m$
  is **doubled** -- the membrane is *faster*, and passes a wider band of
  frequencies. At 50 Hz it now attenuates *less* than it did before.

Report your new cut-off frequency below.

> **Careful:** keep `G` doubled only for this question. Set it back to
> `G = g_leak * nS` and re-run Steps 2 to 4 before you re-run any earlier
> submission cell -- Questions 1 to 5 are graded against your **original** leak
> conductance.

In [ ]:
f_c_doubled_answer = None  # <-- replace with your answer, in Hz

assignment.submit(f_c_doubled_answer, "f_cutoff_doubled_g")